# 2.1 · Anatomía de una herramienta

**Objetivo:** entender qué es una herramienta para el modelo (un contrato descrito en lenguaje y JSON Schema) y que **el modelo no ejecuta nada**.

**Al terminar tendrás:** una vuelta completa `tool_use` → `tool_result` hecha a mano, una celda por paso.

## Preparación

La celda de siempre, más `json` (los resultados de las herramientas se devuelven como texto JSON).

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import anthropic
client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

In [ ]:
SYSTEM = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, de forma breve y basándote solo en los datos que tengas."
)

## Sin herramientas, el modelo no sabe

Hasta ahora Claude solo hablaba. Si le preguntamos por datos que no tiene, o se los inventa o dice que no los conoce.

In [ ]:
response = client.messages.create(
    model=MODEL, max_tokens=1024, system=SYSTEM,
    messages=[{"role": "user", "content": "¿Cuántas incidencias abiertas tiene ACME?"}],
)
print(next(b.text for b in response.content if b.type == "text"))

Los datos existen: están en `datos/incidencias.json` y `herramientas.py` tiene las funciones para consultarlos. No son el foco del taller, así que ya están escritas.

In [ ]:
from herramientas import buscar_incidencias, crear_incidencia

buscar_incidencias(cliente="ACME", estado="abierta")

La función existe y funciona. Falta que el modelo pueda **pedir** que la ejecutemos.

## Definir la herramienta

Para el modelo, una herramienta es una descripción con tres partes:

| Parte | Para qué sirve |
|---|---|
| `name` | cómo la pide el modelo |
| `description` | lo que el modelo **lee para decidir** si la usa y cuándo |
| `input_schema` | qué parámetros puede pasar, en JSON Schema (tipos, `enum`, obligatorios) |

**La descripción es un prompt.** Es lo más importante de la herramienta: dice cuándo usarla, qué devuelve y qué pasa si no hay resultados. En el bloque 5 lo comprobaremos rompiéndola.

Fíjate en que aquí no hay ni una línea de Python: el modelo nunca ve el código de `buscar_incidencias`, solo esta descripción.

In [ ]:
TOOLS = [{
    "name": "buscar_incidencias",
    "description": (
        "Busca incidencias de soporte en la base de datos. Úsala siempre que el usuario pregunte "
        "por incidencias existentes, su estado o las de un cliente concreto. Devuelve una lista "
        "de incidencias con id, cliente, asunto, estado y prioridad. Si no hay coincidencias devuelve []."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "texto":   {"type": "string", "description": "Texto libre a buscar en asunto y descripción"},
            "estado":  {"type": "string", "enum": ["abierta", "cerrada"]},
            "cliente": {"type": "string", "description": "Nombre exacto del cliente"},
        },
        "required": [],
    },
}]

## Primera llamada con herramientas

La misma pregunta, ahora pasando `tools=TOOLS`. Guardamos la conversación en `messages` porque la vamos a continuar.

In [ ]:
messages = [{"role": "user", "content": "¿Cuántas incidencias abiertas tiene ACME?"}]
response = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM,
                                  tools=TOOLS, messages=messages)
response.stop_reason

`stop_reason` es `tool_use`: el modelo se ha detenido para pedir algo. Lo que pide está en un bloque de tipo `tool_use`, con un `id`, el `name` de la herramienta y el `input` con los parámetros.

In [ ]:
tool_use = next(b for b in response.content if b.type == "tool_use")
tool_use

**No ha buscado nada. Me está pidiendo que yo busque.** El modelo propone; mi código ejecuta.

## Cerrar la vuelta a mano

Cuatro pasos, una celda por paso. Esto es lo que en el siguiente vídeo hará un bucle.

In [ ]:
# Paso 1: YO ejecuto lo que el modelo ha pedido
resultado = buscar_incidencias(**tool_use.input)
resultado

In [ ]:
# Paso 2: guardo en el historial lo que pidió el modelo
messages.append({"role": "assistant", "content": response.content})

In [ ]:
# Paso 3: le devuelvo el resultado, enlazado por id
messages.append({"role": "user", "content": [{
    "type": "tool_result",
    "tool_use_id": tool_use.id,
    "content": json.dumps(resultado, ensure_ascii=False),
}]})
messages

En `messages` se ve la secuencia completa: `user` (la pregunta) → `assistant` (el `tool_use`) → `user` (el `tool_result`).

- El resultado va en un mensaje **`user`**: desde el punto de vista del modelo, es información que le llega de fuera.
- Se enlaza con la petición por `tool_use_id`: así el modelo sabe a qué petición responde cada resultado.

In [ ]:
# Paso 4: vuelvo a llamar con todo el historial
response = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM,
                                  tools=TOOLS, messages=messages)
response.stop_reason, next(b.text for b in response.content if b.type == "text")

Ahora `stop_reason` es `end_turn` y la respuesta se basa en datos reales.

## Resumen

- Una herramienta es un contrato: **nombre**, **descripción** (un prompt) y **esquema**.
- Con herramientas, el modelo puede parar con `stop_reason == "tool_use"` y un bloque `tool_use`: **pide**, no ejecuta.
- La vuelta se cierra en el código: ejecutar, guardar el turno `assistant`, devolver un `tool_result` con el mismo `tool_use_id` y volver a llamar.

**Para el siguiente vídeo:** ¿y si necesita buscar, luego crear y luego volver a buscar? Estas cuatro celdas tienen que repetirse solas. Necesitamos un bucle.

### Errores habituales

- **No añadir el turno `assistant` antes del `tool_result`:** la API responde con un error 400, porque el resultado no tiene una petición previa a la que responder.
- **Descripciones de una sola palabra** (`"Busca incidencias."`): el modelo no sabe cuándo usar la herramienta ni qué esperar de ella.